# 实验报告03：正则表达式与网络结构化数据

学号：20244080115　姓名：胡勇彦

In [1]:
import hashlib
import os
import platform
import uuid
import sys

STUDENT_ID = 20244080115

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()} | {platform.machine()} | {platform.system()} | {uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID} - {MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python] {sys.version.split()[0]}")

[Conda 环境] base
[学号] 20244080115
[机器指纹] aca91e3578be51ad
[RUN_ID] 20244080115 - aca91e3578be51ad
[Python] 3.14.7


## 实验1：正则表达式的基础匹配

In [2]:
import re

### 1. 在示例文本上匹配 `\d+`、`\s+`、`\bcat\b`

In [3]:
text1 = "cat on mat; later: cats and dogs"
print(r"\d+ :", re.findall(r"\d+", text1))
print(r"\s+ :", re.findall(r"\s+", text1))
print(r"\bcat\b :", re.findall(r"\bcat\b", text1))

\d+ : []
\s+ : [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b : ['cat']


**说明：** 文本中没有数字，所以 `\d+` 匹配为空；`\s+` 匹配每段连续空白；`\bcat\b` 用词边界只匹配独立单词 cat，`cats` 中的 cat 不匹配。

### 2. `cat|dog` 交替匹配，以及交替顺序导致的差异

In [4]:
text2 = "cat dog bird CAT"
print("cat|dog:", re.findall(r"cat|dog", text2))
print("不区分大小写:", re.findall(r"cat|dog", text2, flags=re.IGNORECASE))
print(r"a|a|a|aaa on aaaa :", re.findall(r"a|a|a|aaa", "aaaa"))
print(r"aaa|a|a on aaaa :", re.findall(r"aaa|a|a", "aaaa"))

cat|dog: ['cat', 'dog']
不区分大小写: ['cat', 'dog', 'CAT']
a|a|a|aaa on aaaa : ['a', 'a', 'a', 'a']
aaa|a|a on aaaa : ['aaa', 'a']


**说明：** 正则交替从左到右尝试，且在每个位置只返回第一个成功的分支。第一个模式分支 `a` 排在前面，每个位置都先匹配单个 `a`，所以得到 4 个单字符；第二个模式 `aaa` 排在前面，会先尽量匹配三字符，再匹配剩余的单字符，所以得到 `aaa` + `a`。分支顺序会改变匹配结果。

### 3. 各量词在 `ct cat caat caaat` 上的匹配

In [5]:
text3 = "ct cat caat caaat"
for pat in [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]:
    print(pat, "->", re.findall(pat, text3))

ca*t -> ['ct', 'cat', 'caat', 'caaat']
ca+t -> ['cat', 'caat', 'caaat']
ca?t -> ['ct', 'cat']
ca{2}t -> ['caat']
ca{1,2}t -> ['cat', 'caat']


**量词含义：**
- `ca*t`：`*` 表示 a 出现 0 次或多次，所以 ct、cat、caat、caaat 全部匹配；
- `ca+t`：`+` 表示 a 出现 1 次或多次，ct 不匹配，其余匹配；
- `ca?t`：`?` 表示 a 出现 0 次或 1 次，只匹配 ct、cat；
- `ca{2}t`：a 恰好出现 2 次，只匹配 caat；
- `ca{1,2}t`：a 出现 1 到 2 次，匹配 cat、caat。

## 实验2：分组、回溯引用与替换

### 1. 捕获组提取邮箱的用户名和域名

In [6]:
mail_text = "联系 : alice@example.com ; 备选 : bob@test.org"
pattern = r"([A-Za-z0-9_.+-]+)@([A-Za-z0-9-]+\.[A-Za-z0-9.-]+)"
for m in re.finditer(pattern, mail_text):
    print("完整邮箱:", m.group(0), "| 用户名:", m.group(1), "| 域名:", m.group(2))

完整邮箱: alice@example.com | 用户名: alice | 域名: example.com
完整邮箱: bob@test.org | 用户名: bob | 域名: test.org


### 2. 后向引用找出连续重复的单词

In [7]:
repeat_text = "cat cat, dog cat, bird bird"
word_repeats = re.findall(r"\b(\w+)\s+\1\b", repeat_text)
print("连续重复的单词:", word_repeats)
for m in re.finditer(r"\b(\w+)\s+\1\b", repeat_text):
    print("匹配片段:", m.group(0))

连续重复的单词: ['cat', 'bird']
匹配片段: cat cat
匹配片段: bird bird


**说明：** `\1` 是后向引用，代表第 1 个捕获组之前匹配到的内容，因此只有相邻且完全相同的单词才会被匹配。

### 3. 日期从“年-月-日”转换为“月-日-年”，保留行数

In [8]:
date_text = "2024-05-01\n2024/05/02\n2024.05.03"
print("转换前:\n", date_text)
converted = re.sub(r"(\d{4})[-/.](\d{2})[-/.](\d{2})", r"\2-\3-\1", date_text)
print("转换后:\n", converted)

转换前:
 2024-05-01
2024/05/02
2024.05.03
转换后:
 05-01-2024
05-02-2024
05-03-2024


### 4. 电话号码统一为 `(123) 456-7890` 形式

In [9]:
phone_text = "联系我：123-456-7890 或 987.654.3210"
normalized = re.sub(r"(\d{3})[-.](\d{3})[-.](\d{4})", r"(\1) \2-\3", phone_text)
print(normalized)

联系我：(123) 456-7890 或 (987) 654-3210


## 实验3：使用 BeautifulSoup 解析 HTML

### 1. 构建解析树，输出页面标题与 h1

In [10]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式
<p class="course">第七讲：结构化数据
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")
print("title 文本:", soup.title.string)
print("h1 标签名:", soup.h1.name, "| h1 文本:", soup.h1.string)

title 文本: 实验课程
h1 标签名: h1 | h1 文本: 数据分析


### 2. `find_all` 找出所有 class="course" 段落和所有链接

In [11]:
paragraphs = soup.find_all("p", class_="course")
links = soup.find_all("a")
for p in paragraphs:
    print("段落:", p.get_text(strip=True))
for a in links:
    print("链接文字:", a.string, "| href:", a["href"])

段落: 第六讲：正则表达式第七讲：结构化数据讲义六讲义七
段落: 第七讲：结构化数据讲义六讲义七
链接文字: 讲义六 | href: https://example.org/lecture06
链接文字: 讲义七 | href: https://example.org/lecture07


### 3. `get_text` 提取可见文本并比较范围

In [12]:
print("整个文档 get_text:")
print(soup.get_text(separator=" | ", strip=True))
print("单个段落 get_text:", paragraphs[0].get_text(strip=True))
print("单个链接 get_text:", links[0].get_text(strip=True))

整个文档 get_text:
实验课程 | 数据分析 | 第六讲：正则表达式 | 第七讲：结构化数据 | 讲义六 | 讲义七
单个段落 get_text: 第六讲：正则表达式第七讲：结构化数据讲义六讲义七
单个链接 get_text: 讲义六


**比较：** 对整个文档调用 `get_text` 会去除所有标签、拼接全部可见文本；对单个段落或链接调用则只返回该元素内部的文本，范围由调用对象决定。